# AEF trading simulation: a first submission in Python

This notebook goes through one decision: see what you may trade, download prices, build a portfolio, check it and submit it. Run it cell by cell; nothing is sent until the `submit()` cell.

## Setup

Install once, in a terminal in your project folder, with [uv](https://docs.astral.sh/uv/):

```sh
uv add "git+https://github.com/advanced-empirical-finance/aefsim#subdirectory=python" tidyfinance
```

Save your team's token once: `aefsim.set_token()` asks for it, asks where to save it and offers to add the file to `.gitignore`. Never paste the token into a notebook you share.

In [ ]:
import datetime as dt

import aefsim
import polars as pl
import tidyfinance as tf

aefsim.set_token()  # once per project

## What can we trade, and under which rules?

In [ ]:
universe = aefsim.get_universe()
close_date = universe["close_date"].max()  # the latest close
print(close_date)
universe.group_by("sp500").len()

`get_universe()` lists the stocks you can buy: common stocks on NYSE, NYSE American and Nasdaq that closed at $5 or more. A stock you hold that drops below $5 stays in your portfolio; you may keep or reduce it, but not buy more.

In [ ]:
aefsim.get_parameters()  # costs, shorting, leverage, position cap; applies_next marks the current rules

In [ ]:
aefsim.get_calendar().filter(pl.col("is_next"))  # the next trading day and its deadline, 09:00 New York time

## Prices

The simulation does not serve prices. tidyfinance downloads them from Yahoo Finance, the same source the simulation trades at. Like aefsim, it returns polars DataFrames.

In [ ]:
candidates = universe.filter(pl.col("sp500"))["ticker"].to_list()
prices = tf.download_data(
    domain="stock_prices",
    symbols=candidates,
    start_date=str(close_date - dt.timedelta(days=365)),
    end_date=str(close_date),
)

## A simple rule

As an example (not a recommendation): the 20 S&P 500 stocks with the lowest volatility over the past year, equal weight.

In [ ]:
volatility = (
    prices.sort("symbol", "date")
    .with_columns(ret=pl.col("adjusted_close").pct_change().over("symbol"))
    .group_by("symbol")
    .agg(vol=pl.col("ret").std() * 252 ** 0.5, days=pl.len())
    .filter(pl.col("days") >= 200)
    .sort("vol")
    .head(20)
)
weights = volatility.select(ticker="symbol", weight=pl.lit(1 / volatility.height))
weights

Weights are fractions of your portfolio's value, in a DataFrame with the columns `ticker` and `weight`. Tickers you leave out get zero, so this replaces your whole portfolio; whatever does not add up to one stays in cash.

## Check, then submit

In [ ]:
check = aefsim.validate(weights, close_date=close_date)
check

`validate()` runs every check the simulation runs and stores nothing. When it says OK, submit. The weights trade at the next market open whose deadline (09:00 New York time, usually 15:00 in Copenhagen) has not passed; the check shows that date. The rationale is your decision log: write down what you expect and why.

In [ ]:
result = aefsim.submit(
    weights,
    close_date=close_date,
    rationale="Low-volatility tilt: equal weight in the 20 least volatile S&P 500 stocks over the past year.",
)
result

Changed your mind? Submit again (the last accepted submission before the deadline is the one that trades), or `aefsim.cancel(result.submission_id)`.

## The next morning

Trades happen at the US open; results appear after the nightly update.

In [ ]:
aefsim.get_submissions()

In [ ]:
aefsim.get_portfolio().explode("positions").unnest("positions")  # one row per position

In [ ]:
aefsim.get_performance()["summary"]

In [ ]:
aefsim.get_leaderboard()